# Análisis de resultados — Modelo ARIMA semanal

Se analizan los resultados del modelo ARIMA semanal comparado con los benchmarks BASE (naive t-1) y ZEROS.
Los resultados se cargan desde los CSV generados por el notebook de modelo.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

## Carga de resultados

In [ ]:
df_res = pd.read_csv('../../Datos_csv/resultados_arima_semanal.csv')
print(f'ETFs cargados: {len(df_res)}')
display(df_res.head())

## Definición de grupos y enriquecimiento del DataFrame

In [ ]:
GRUPOS_ETF = {
    'Mercado estadounidense (Core US)': ['SPY','IVV','VOO','QQQ','DIA','IWM','MDY','IJR','ITOT','VTI'],
    'Factores y estilos': ['MTUM','QUAL','USMV','VLUE','IWF','IWD','VUG','VTV','VIG','DVY','SCHD'],
    'Sectores económicos': ['XLK','XLF','XLV','XLY','XLP','XLE','XLI','XLB','XLU','XLRE','VNQ'],
    'Mercados internacionales': ['EFA','IEFA','VEA','EEM','IEMG','VWO','EWJ','EWG','EWQ','EWU','EWT','EWZ','FXI','MCHI','INDA'],
    'Renta fija': ['AGG','BND','LQD','HYG','JNK','TLT','IEF','SHY','TIP'],
    'Materias primas': ['GLD','IAU','SLV','USO','DBC']
}

etf_to_grupo = {}
for grupo, tickers in GRUPOS_ETF.items():
    for t in tickers:
        etf_to_grupo[t] = grupo

df_anal = df_res.copy()
df_anal['Grupo'] = df_anal['ETF'].map(etf_to_grupo).fillna('Otros')
df_anal['Mejora_RMSE_vs_base']  = (df_anal['RMSE_baseline'] - df_anal['RMSE'])  / df_anal['RMSE_baseline'] * 100
df_anal['Mejora_RMSE_vs_zeros'] = (df_anal['RMSE_zeros']    - df_anal['RMSE'])  / df_anal['RMSE_zeros']    * 100
df_anal['Bate_baseline'] = (df_anal['RMSE'] < df_anal['RMSE_baseline']).astype(int)
df_anal['Bate_zeros']    = (df_anal['RMSE'] < df_anal['RMSE_zeros']).astype(int)
print('Grupos asignados.')
display(df_anal[['ETF','Grupo','RMSE','RMSE_baseline','RMSE_zeros','Mejora_RMSE_vs_base','Mejora_RMSE_vs_zeros']].head())

## RMSE y MAE medio global — ARIMA semanal

In [ ]:
rmse_m = df_anal['RMSE'].mean()
mae_m  = df_anal['MAE'].mean()
rmse_b = df_anal['RMSE_baseline'].mean()
mae_b  = df_anal['MAE_baseline'].mean()
rmse_z = df_anal['RMSE_zeros'].mean()
mae_z  = df_anal['MAE_zeros'].mean()

tabla_global = pd.DataFrame({
    'Modelo':  ['ARIMA', 'BASE (naive t-1)', 'ZEROS'],
    'RMSE':    [rmse_m, rmse_b, rmse_z],
    'MAE':     [mae_m,  mae_b,  mae_z],
}).set_index('Modelo')

print('='*60)
print('TABLA GLOBAL — Modelo ARIMA semanal')
print('='*60)
display(tabla_global.style.format('{:.6f}').highlight_min(color='lightgreen', axis=0))

## RMSE y MAE por grupos de ETFs — ARIMA semanal

In [ ]:
orden_grupos = list(GRUPOS_ETF.keys())
print('='*100)
print('TABLA POR GRUPOS — Modelo ARIMA semanal')
print('='*100)
for grupo in orden_grupos:
    sub = df_anal[df_anal['Grupo'] == grupo]
    if sub.empty:
        continue
    tabla = pd.DataFrame({
        'Modelo': ['ARIMA', 'BASE', 'ZEROS'],
        'RMSE':   [sub['RMSE'].mean(), sub['RMSE_baseline'].mean(), sub['RMSE_zeros'].mean()],
        'MAE':    [sub['MAE'].mean(),  sub['MAE_baseline'].mean(),  sub['MAE_zeros'].mean()],
    }).set_index('Modelo')
    print(f'\nGrupo: {grupo} ({len(sub)} ETFs)')
    display(tabla.style.format('{:.6f}').highlight_min(color='lightgreen', axis=0))

## Ranking de ETFs — mejora RMSE (%) vs baseline y vs zeros — ARIMA semanal

Cada barra muestra cuánto mejora (o empeora) ARIMA respecto a cada benchmark. Verde = mejora.

In [ ]:
df_plot = df_anal.sort_values('Mejora_RMSE_vs_base')
fig, axes = plt.subplots(1, 2, figsize=(18, max(8, len(df_anal)*0.32)))

for ax, col, titulo in zip(
    axes,
    ['Mejora_RMSE_vs_base', 'Mejora_RMSE_vs_zeros'],
    ['ARIMA vs BASE (naive t-1)', 'ARIMA vs ZEROS']
):
    colores = ['#4CAF50' if v >= 0 else '#F44336' for v in df_plot[col]]
    ax.barh(df_plot['ETF'], df_plot[col], color=colores)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Mejora RMSE (%)')
    ax.set_title(titulo, fontsize=11)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Ranking de ETFs — ARIMA semanal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Distribución de mejora de RMSE por grupo de activos — ARIMA semanal

In [ ]:
etiq_g = {
    'Mercado estadounidense (Core US)': 'Core US',
    'Factores y estilos': 'Factores',
    'Sectores económicos': 'Sectores',
    'Mercados internacionales': 'Internacional',
    'Renta fija': 'Renta fija',
    'Materias primas': 'Materias primas'
}
df_anal['Grupo_corto'] = df_anal['Grupo'].map(etiq_g).fillna('Otros')

orden_corto = [etiq_g[g] for g in orden_grupos if g in etiq_g]
datos_box = [df_anal[df_anal['Grupo_corto']==g]['Mejora_RMSE_vs_base'].values for g in orden_corto]

fig, ax = plt.subplots(figsize=(12, 5))
bp = ax.boxplot(datos_box, labels=orden_corto, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
colores_box = ['#4FC3F7','#81C784','#FFB74D','#E57373','#CE93D8','#FFF176']
for patch, color in zip(bp['boxes'], colores_box):
    patch.set_facecolor(color)
ax.axhline(0, color='red', linewidth=1, linestyle='--', alpha=0.7)
ax.set_ylabel('Mejora RMSE vs BASE (%)')
ax.set_title('Distribución de mejora de RMSE por grupo — ARIMA semanal', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Porcentaje de ETFs donde ARIMA supera cada benchmark — ARIMA semanal

In [ ]:
n_total = len(df_anal)
pct_b   = df_anal['Bate_baseline'].mean() * 100
pct_z   = df_anal['Bate_zeros'].mean()    * 100

print(f'ETFs totales analizados: {n_total}')
print(f'ARIMA bate BASE   en {pct_b:.1f}% de los ETFs ({int(df_anal["Bate_baseline"].sum())}/{n_total})')
print(f'ARIMA bate ZEROS  en {pct_z:.1f}% de los ETFs ({int(df_anal["Bate_zeros"].sum())}/{n_total})')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['vs BASE', 'vs ZEROS'], [pct_b, pct_z], color=['#4CAF50','#2196F3'], width=0.5)
ax.axhline(50, color='red', linewidth=1, linestyle='--', alpha=0.7)
ax.set_ylabel('% ETFs donde ARIMA gana')
ax.set_ylim(0, 105)
ax.set_title('% ETFs donde ARIMA supera el benchmark — semanal', fontsize=11)
for i, v in enumerate([pct_b, pct_z]):
    ax.text(i, v+1.5, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## Análisis direccional — Kappa de Cohen y Accuracy — ARIMA semanal

Además de las métricas de error, se analiza si ARIMA acierta la dirección del movimiento (subida/bajada).

In [ ]:
df_dir = pd.read_csv('../../Datos_csv/direccional_arima_semanal.csv')
print(f'ETFs con métricas direccionales: {len(df_dir)}')

df_dir['Kappa_ARIMA'] = pd.to_numeric(df_dir['Kappa_ARIMA'], errors='coerce')
df_dir['Kappa_BASE']  = pd.to_numeric(df_dir['Kappa_BASE'],  errors='coerce')

print(f'\nKappa medio ARIMA: {df_dir["Kappa_ARIMA"].mean():.4f}')
print(f'Kappa medio BASE:  {df_dir["Kappa_BASE"].mean():.4f}')

acc_arima = df_dir['Accuracy_ARIMA'].str.replace('%','').astype(float)
acc_base  = df_dir['Accuracy_BASE'].str.replace('%','').astype(float)
print(f'Accuracy media ARIMA: {acc_arima.mean():.2f}%')
print(f'Accuracy media BASE:  {acc_base.mean():.2f}%')

display(df_dir.set_index('ETF').sort_values('Kappa_ARIMA', ascending=False))